In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

root = Path("results")
csv_files = list(root.glob("*/periodic_image_distance/pi_dist_summary_*.csv"))
print(f"Found {len(csv_files)} CSV files.")

dfs = []

for f in csv_files:
    pdbid = f.parts[-3]
    df = pd.read_csv(f)
    df["pdb"] = pdbid
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

display(df_all.head())
print(df_all.columns)
print(df_all["pdb"].unique())
print(df_all.shape)

Found 112 CSV files.


,pdb,replica,xtc,xvg,n_frames,min_dist_nm,max_dist_nm,mean_dist_nm,median_dist_nm,p01_dist_nm,p05_dist_nm,p95_dist_nm,p99_dist_nm,status
0,5s2e,0,cleaned_5s2e_0.xtc,pi_dist_5s2e_0.xvg,10001,1.402,2.296,1.920371,1.932,1.589,1.692,2.109,2.163,ok
1,5s2e,1,cleaned_5s2e_1.xtc,pi_dist_5s2e_1.xvg,10001,1.243,2.291,1.898579,1.919,1.487,1.619,2.108,2.164,ok
2,5s2e,2,cleaned_5s2e_2.xtc,pi_dist_5s2e_2.xvg,10001,1.001,2.250,1.886944,1.912,1.337,1.571,2.104,2.166,ok
3,5s2e,3,cleaned_5s2e_3.xtc,pi_dist_5s2e_3.xvg,10001,1.122,2.274,1.936946,1.948,1.620,1.725,2.115,2.165,ok
4,5s2e,4,cleaned_5s2e_4.xtc,pi_dist_5s2e_4.xvg,10001,1.012,2.259,1.911616,1.932,1.428,1.639,2.111,2.164,ok


Index(['pdb', 'replica', 'xtc', 'xvg', 'n_frames', 'min_dist_nm',
       'max_dist_nm', 'mean_dist_nm', 'median_dist_nm', 'p01_dist_nm',
       'p05_dist_nm', 'p95_dist_nm', 'p99_dist_nm', 'status'],
      dtype='object')
['5s2e' '1ncx' '2bem' '7p8f' '1l0c' '4eqk' '1xyh' '6ghh' '5s3v' '7gzz'
 '4ykw' '5kch' '1vc1' '5b08' '7due' '5wa8' '8okr' '3a8c' '6el5' '8den'
 '7f2b' '7y05' '1hia' '4dg4' '2o3s' '2znt' '8az4' '2gg7' '3m3x' '4rfz'
 '4ded' '7nns' '6bsk' '2aps' '8bpt' '5fns' '2p16' '2fqw' '2c2l' '1yxv'
 '1ptz' '8uut' '7d2d' '1m2r' '8wtr' '8c8s' '5osu' '7ons' '5fp8' '4gub'
 '5c1w' '3giv' '4a39' '6gh1' '8roo' '3t00' '2hla' '2q11' '8erx' '7s5s'
 '8gs9' '4j0j' '6eh8' '1ggp' '7xed' '1fon' '2qhr' '7n6h' '8xuk' '2mcp'
 '5zr8' '2vff' '6b61' '1e65' '3b6d' '9jdq' '6e4d' '1fof' '9mjl' '9j5z'
 '1ydv' '8ifj' '1a7u' '3b9c' '1yvx' '1zty' '3d6d' '6qpm' '8bdp' '7wpi'
 '4dme' '3nxb' '5j7e' '1wy9' '3ef4' '7lak' '3era' '1onl' '4hoi' '7n3z'
 '1nbq' '8skm' '1kap' '4azu' '8spm' '3h90' '1dxx' '4xqa' '5rsi' '1p2

In [3]:
# sort for min dsitance
cutoff = 1.2 #nm

bad = df_all["too_small"] = df_all["min_dist_nm"] < cutoff
safe = df_all["safe"] = df_all["min_dist_nm"] >= cutoff + 0.1
marginal = df_all["marginal"] = (df_all["min_dist_nm"] > cutoff) & (df_all["min_dist_nm"] < cutoff + 0.1) 

flagged = df_all[~df_all["safe"]]

print(f"Flagged {flagged.shape[0]} out of {df_all.shape[0]} structures ({flagged.shape[0]/df_all.shape[0]*100:.2f}%)")
display(flagged[["pdb", "min_dist_nm"]])

Flagged 928 out of 1118 structures (83.01%)


,pdb,min_dist_nm
1,5s2e,1.243
2,5s2e,1.001
3,5s2e,1.122
4,5s2e,1.012
5,5s2e,1.117
...,...,...
1113,2vfd,1.230
1114,2vfd,1.259
1115,2vfd,1.098
1116,2vfd,1.110


In [4]:
#filter for PDB only

flagged_pdbs = flagged["pdb"].unique()
print(f"Flagged {len(flagged_pdbs)} unique PDBs.")
print(flagged_pdbs)

Flagged 109 unique PDBs.
['5s2e' '1ncx' '2bem' '7p8f' '1l0c' '4eqk' '1xyh' '6ghh' '5s3v' '7gzz'
 '4ykw' '5kch' '1vc1' '5b08' '7due' '5wa8' '8okr' '3a8c' '6el5' '8den'
 '7f2b' '7y05' '1hia' '4dg4' '2o3s' '2znt' '8az4' '2gg7' '3m3x' '4rfz'
 '4ded' '7nns' '6bsk' '2aps' '8bpt' '5fns' '2p16' '2fqw' '2c2l' '1yxv'
 '1ptz' '8uut' '7d2d' '1m2r' '8wtr' '8c8s' '5osu' '7ons' '5fp8' '4gub'
 '5c1w' '3giv' '4a39' '6gh1' '8roo' '3t00' '2hla' '2q11' '8erx' '7s5s'
 '4j0j' '6eh8' '1ggp' '7xed' '1fon' '2qhr' '8xuk' '2mcp' '5zr8' '2vff'
 '6b61' '1e65' '3b6d' '9jdq' '6e4d' '1fof' '9mjl' '9j5z' '1ydv' '8ifj'
 '1a7u' '3b9c' '1yvx' '1zty' '3d6d' '6qpm' '8bdp' '7wpi' '4dme' '3nxb'
 '5j7e' '1wy9' '3ef4' '3era' '1onl' '4hoi' '7n3z' '1nbq' '8skm' '1kap'
 '4azu' '8spm' '3h90' '1dxx' '4xqa' '5rsi' '1p27' '7bru' '2vfd']


In [6]:
#group by pdb and get the min for each pdb

grouped = df_all.groupby("pdb").agg({"min_dist_nm": "min", "too_small": "any", "safe": "any", "marginal": "any"}).reset_index()
display(grouped.head())


,pdb,min_dist_nm,too_small,safe,marginal
0,1a7u,0.237,True,False,True
1,1dxx,0.296,True,False,False
2,1e65,0.935,True,True,True
3,1fof,1.109,True,True,True
4,1fon,0.276,True,False,False


In [18]:
#take all options below a set value

filtered = df_all[df_all["min_dist_nm"] < 0.4]
print(f"Found {filtered.shape[0]} options with min distance < 0.4 nm.")
display(filtered[["pdb", "min_dist_nm", "xtc"]])
if not (root / "000analysis").exists():
    (root / "000analysis").mkdir(parents=True)
filtered.to_csv(root / "000analysis" / "pi_dist_filtered.csv", index=False)

Found 108 options with min distance < 0.4 nm.


,pdb,min_dist_nm,xtc
67,1xyh,0.332,cleaned_1xyh_7.xtc
121,1vc1,0.325,cleaned_1vc1_1.xtc
123,1vc1,0.253,cleaned_1vc1_3.xtc
125,1vc1,0.336,cleaned_1vc1_5.xtc
127,1vc1,0.263,cleaned_1vc1_7.xtc
...,...,...,...
1052,3h90,0.252,cleaned_3h90_4.xtc
1053,3h90,0.381,cleaned_3h90_5.xtc
1056,3h90,0.299,cleaned_3h90_8.xtc
1062,1dxx,0.296,cleaned_1dxx_4.xtc


In [25]:
# group by pdbid, get the min value of mindist per each pdbid
filtered = filtered.sort_values("min_dist_nm")
grouped_filtered = filtered.groupby("pdb").agg({"min_dist_nm": "min", "xtc": "first"}).reset_index()
display(grouped_filtered.head())
print(grouped_filtered.shape)

#order them in ascending order of min_dist_nm and save to csv
grouped_filtered = grouped_filtered.sort_values("min_dist_nm")
grouped_filtered.to_csv(root / "000analysis" / "pi_dist_filtered_grouped.csv", index=False)



,pdb,min_dist_nm,xtc
0,1a7u,0.237,cleaned_1a7u_7.xtc
1,1dxx,0.296,cleaned_1dxx_4.xtc
2,1fon,0.276,cleaned_1fon_1.xtc
3,1kap,0.253,cleaned_1kap_1.xtc
4,1nbq,0.236,cleaned_1nbq_3.xtc


(27, 3)
